# FlashAttention: Making Attention Hardware-Friendly

> In the inference section, we mentioned that FlashAttention splits Q, K, V into small blocks and computes them inside SRAM, avoiding repeated round-trips of the huge attention matrix to GPU memory. At the time, we glossed over it with a single "mathematically equivalent" sentence, without explaining why it can be both fast and exact.
>
> This appendix starts from the most basic numerical stability, derives online softmax and tiling step by step, and finally implements a teaching version in PyTorch that can be cross-checked against the official `scaled_dot_product_attention`.


The standard attention formula fits on one line: $O = \mathrm{softmax}(QK^\top / \sqrt{d}) V$. The formula is concise, but a naive implementation suffers from two problems. The first is numerical: elements of $QK^\top$ can be very large, causing $e^{x}$ to overflow. The second is memory traffic: the $N \times N$ attention matrix must first be written to HBM, then read back to compute the softmax and the weighted sum, so the read/write volume grows quadratically with sequence length.

FlashAttention solves both problems with two core tricks. The first is online softmax: it fuses the "find max" and "sum" passes into a single scan, maintaining three running variables $(m, d, o)$ along the way. The second is tiling: it splits Q, K, V into blocks, loads only one block of K, V into SRAM at a time, updates the running variables after each block, and never writes intermediate results back to HBM.

The whole process introduces no approximation. The final output is mathematically identical to standard attention, differing only at the $10^{-6}$ level of floating point error. This is the fundamental difference between FlashAttention and sparse attention or linear attention — the latter two trade away exactness to save compute, while FlashAttention merely reorders the computation to save memory traffic.


In [ ]:
# This appendix uses only torch and numpy, no high-level attention API
import math
import numpy as np
import torch
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print(f"torch: {torch.__version__}")


## 1. The Two Tiers of GPU Memory: HBM and SRAM

To understand what FlashAttention accelerates, we first need to look at the memory hierarchy of a GPU. A modern GPU (such as the A100) has two tiers of memory:

| Tier | Capacity | Bandwidth | Role |
|:---|:---|:---|:---|
| HBM (high-bandwidth memory) | 80 GB | ~2 TB/s | Main memory, where all tensors live |
| SRAM (on-chip cache) | ~20 MB | ~19 TB/s | Private to each SM, ~10x faster |

Capacity differs by 4000x and bandwidth by nearly 10x. This means: a single matrix multiplication is fast, but moving data from HBM to SRAM is slow. If a computation repeatedly shuttles the same data between HBM and SRAM, the transfer time will far exceed the compute time.

This working pattern is called memory-bound. Attention is a classic memory-bound operator.


### What Standard Attention Is Moving Around

The forward pass of standard attention has four steps:

1. Compute $S = QK^\top$, producing the $N \times N$ attention score matrix, **write back to HBM**
2. Read $S$ back from HBM, compute row max, write back to HBM
3. Read $S$ and max back from HBM, compute softmax to get $P$, **write back to HBM**
4. Read $P$ and $V$ back from HBM, compute $O = PV$, write back to HBM

Every $N \times N$ matrix is read and written to HBM multiple times. Let's get a feel for the scale with concrete numbers.


In [ ]:
# Memory footprint of the standard attention matrix at different sequence lengths (FP16, 2 bytes/element)
def attn_matrix_bytes(seq, head_dim=128, n_heads=32, bytes_per_elem=2):
    # Intermediate matrices written to HBM per forward pass: S (attention scores) and P (after softmax)
    # One copy per head, per batch
    elems = n_heads * seq * seq
    return elems * bytes_per_elem

def fmt(b):
    if b >= 1024**3:
        return f"{b/1024**3:.2f} GB"
    if b >= 1024**2:
        return f"{b/1024**2:.1f} MB"
    return f"{b/1024:.1f} KB"

print(f"{'seq len':>8}  {'one S/P matrix':>15}  {'5 read/writes total':>20}")
print("-" * 55)
for seq in [512, 1024, 2048, 4096, 8192, 16384, 32768]:
    single = attn_matrix_bytes(seq)
    # S written 1x, read 2x (for max, for softmax); P written 1x, read 1x
    traffic = single * 5
    print(f"{seq:>8d}  {fmt(single):>15}  {fmt(traffic):>20}")

print()
print("At seq=32768, moving the attention matrices alone requires on the order of 1 TB of data.")
print("The A100's HBM bandwidth is 2 TB/s, so the transfer alone takes about 0.5 seconds — far slower than the compute itself.")


## 2. Numerical Stability: Why Softmax Blows Up

Before diving into online softmax, let's address a more basic problem: the softmax formula $\frac{e^{x_i}}{\sum_j e^{x_j}}$ blows up if computed directly.

The exponential $e^x$ grows extremely fast. Under FP32, $e^{88}$ or so already overflows to inf. In attention, each element of $QK^\top$ is the dot product of two $d$-dimensional vectors, and with $d=128$ a single element exceeding 100 is common.


In [ ]:
# Demonstrate the overflow threshold of exp under FP32
import torch

x = torch.tensor([80.0, 85.0, 88.0, 90.0, 100.0])
print("x       exp(x)")
for xi in x:
    print(f"{xi.item():>6.1f}  {torch.exp(xi).item():>15.3e}")

print()
print("Under FP32, exp(89) is already close to inf.")

# Simulate the magnitude of an attention score
d_head = 128
q = torch.randn(d_head)
k = torch.randn(d_head)
score = (q * k).sum() / math.sqrt(d_head)
print(f"With random init, a d=128 attention score is on the order of ±{abs(score.item()):.1f}")
print("But late in training, q and k norms can grow 5-10x, pushing the score above 80.")


### 2.1 Stable Softmax: Subtract the Max

Softmax has a key property: **shifting the entire input by a constant $c$ leaves the output unchanged**.

$$
\mathrm{softmax}(x - c)_i = \frac{e^{x_i - c}}{\sum_j e^{x_j - c}} = \frac{e^{x_i} \cdot e^{-c}}{e^{-c} \sum_j e^{x_j}} = \mathrm{softmax}(x)_i
$$

$e^{-c}$ appears in both the numerator and denominator and cancels out. Set $c$ to $x_\max$ and the largest exponent becomes $e^0 = 1$, so every exponent is $\le 1$ and there is no overflow risk.

This is stable softmax: scan once to find the max, then scan again to compute $\frac{e^{x_i - x_\max}}{\sum_j e^{x_j - x_\max}}$.


In [ ]:
def naive_softmax(x):
    """Direct implementation, will overflow"""
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum()

def stable_softmax(x):
    """Subtract max for numerical stability"""
    m = x.max()
    exp_x = torch.exp(x - m)
    return exp_x / exp_x.sum()

# Construct an input that makes the naive version blow up
x = torch.tensor([1.0, 2.0, 3.0]) * 50  # max is 150
print("input:", x.tolist())
print("naive softmax:", naive_softmax(x).tolist())
print("stable softmax:", stable_softmax(x).tolist())

print()
print("In the naive version, exp(150) overflows to inf and the result becomes nan.")
print("In the stable version, max=150 is subtracted away, so the largest exponent is exp(0)=1 — safe.")


### 2.2 The logsumexp Trick

We often need to compute $\log \sum_i e^{x_i}$ (for example, in cross-entropy loss), and it blows up even more easily than softmax. Computing $\sum_i e^{x_i}$ directly also overflows.

Apply the stable softmax idea in reverse: factor the max out inside the $\log$.

$$
\log \sum_i e^{x_i} = \log \left( e^{x_\max} \sum_i e^{x_i - x_\max} \right) = x_\max + \log \sum_i e^{x_i - x_\max}
$$

After subtracting the max, the largest exponential term is $e^0 = 1$, the sum is at least 1, and the $\log$ never hits zero. This is the logsumexp trick — nearly every deep learning framework's `log_softmax` is built on it.


In [ ]:
def naive_logsumexp(x):
    """Direct log(sum(exp(x))), will overflow"""
    return torch.log(torch.exp(x).sum())

def stable_logsumexp(x):
    """logsumexp trick"""
    m = x.max()
    return m + torch.log(torch.exp(x - m).sum())

x = torch.tensor([100.0, 101.0, 102.0])
print("input:", x.tolist())
print("naive logsumexp:", naive_logsumexp(x).item())
print("stable logsumexp:", stable_logsumexp(x).item())
print("official torch: ", torch.logsumexp(x, dim=0).item())

print()
print("These three should be exactly equal (except the naive one blows up to nan).")
print("This trick will reappear repeatedly in online softmax —")
print("FlashAttention's running variables are essentially a streaming version of logsumexp.")


## 3. Online Softmax: Fusing Two Passes Into One

Stable softmax solves the overflow problem, but at the cost of scanning the data twice: once to find $x_\max$ and once to compute $\sum_j e^{x_j - x_\max}$. On a GPU, two passes mean reading HBM twice.

FlashAttention's first core trick comes from an observation: **you can absolutely find the max and accumulate the denominator at the same time**, as long as you rescale the accumulated value whenever a larger max appears.

This is online softmax.


### 3.1 Two-Pass vs Single-Pass

The two-pass flow of classic stable softmax:

```text
Pass 1: scan x_1, ..., x_N, record m = max
Pass 2: scan x_1, ..., x_N, accumulate d = sum exp(x_i - m)
Return: softmax_i = exp(x_i - m) / d
```

Online softmax changes this to a single pass:

```text
Scan x_1, ..., x_N while maintaining:
  m_k = max(x_1, ..., x_k)            # running max
  d_k = sum_{j=1}^k exp(x_j - m_k)    # running denominator (shifted by current max)

For each new x_{k+1}:
  1. update max: m_{k+1} = max(m_k, x_{k+1})
  2. rescale old sum: d_{k+1} = d_k * exp(m_k - m_{k+1}) + exp(x_{k+1} - m_{k+1})
```

The key is the second step: when the max grows from $m_k$ to $m_{k+1}$, all previous $\exp(x_j - m_k)$ terms need to be re-benchmarked against $m_{k+1}$. But $\exp(x_j - m_{k+1}) = \exp(x_j - m_k) \cdot \exp(m_k - m_{k+1})$, so multiplying $d_k$ by $\exp(m_k - m_{k+1})$ completes the rescale in one shot.

When the max does not change ($m_k = m_{k+1}$), $\exp(m_k - m_{k+1}) = \exp(0) = 1$, the old sum is untouched, and we simply add the new term.


### 3.2 Walking Through by Hand

Use $x = [3, 1, 2]$ as a demo. The final stable softmax is $\frac{[e^3, e^1, e^2]}{e^3 + e^1 + e^2}$, with max=3.

Step through the online flow:

| Step | $x_{k+1}$ | new max $m$ | rescale factor $e^{m_\text{old} - m_\text{new}}$ | new denom $d$ |
|:---|:---|:---|:---|:---|
| init | - | $-\infty$ | - | 0 |
| see $x_1=3$ | 3 | $\max(-\infty, 3) = 3$ | $e^{-\infty - 3} = 0$ | $0 \cdot 0 + e^{3-3} = 1$ |
| see $x_2=1$ | 1 | $\max(3, 1) = 3$ | $e^{3-3} = 1$ | $1 \cdot 1 + e^{1-3} = 1 + 0.135 = 1.135$ |
| see $x_3=2$ | 2 | $\max(3, 2) = 3$ | $e^{3-3} = 1$ | $1.135 \cdot 1 + e^{2-3} = 1.135 + 0.368 = 1.503$ |

Final $m=3$, $d=1.503$. Check: $e^3 + e^1 + e^2 = 20.09 + 2.72 + 7.39 = 30.20$, and $d \cdot e^m = 1.503 \cdot e^3 = 1.503 \cdot 20.09 = 30.20$. The two are equal, which shows that the $(m, d)$ computed online matches the classic algorithm exactly.

Note that the whole process scans $x$ only once, and each step uses only the current element and the three running variables.


In [ ]:
def online_softmax(x):
    """Single-pass softmax, maintains running (m, d)"""
    m = float('-inf')   # running max
    d = 0.0             # running denominator sum exp(x_i - m)

    for xi in x:
        m_new = max(m, xi.item())
        # rescale the old sum and add the new term
        d = d * math.exp(m - m_new) + math.exp(xi.item() - m_new)
        m = m_new

    # Second pass: use the final (m, d) to compute each position's softmax
    # FlashAttention folds this step into the forward pass via tiling
    return torch.tensor([math.exp(xi.item() - m) / d for xi in x])

x = torch.tensor([3.0, 1.0, 2.0])
print("input:", x.tolist())
print("online softmax:    ", online_softmax(x).tolist())
print("stable softmax:    ", stable_softmax(x).tolist())
print("official torch softmax:", torch.softmax(x, dim=0).tolist())

print()
print("All three are numerically identical — the online flow introduces no approximation.")


## 4. From Softmax to the Attention Output

The final attention output is not softmax itself, but $o = \sum_i p_i v_i$, where $p_i$ is the softmax weight and $v_i$ is the value vector. Expanding $p_i$:

$$
o = \sum_i \frac{e^{x_i}}{\sum_j e^{x_j}} v_i = \frac{\sum_i e^{x_i} v_i}{\sum_i e^{x_i}}
$$

Multiply numerator and denominator by $e^{-x_\max}$ to cancel the overflow:

$$
o = \frac{\sum_i e^{x_i - x_\max} v_i}{\sum_i e^{x_i - x_\max}}
$$

Here $x_i = q \cdot k_i / \sqrt{d}$ is the dot product of the query with the $i$-th key, and $v_i$ is the $i$-th value. In addition to $(m, d)$, we maintain a running output $o_k = \sum_{i=1}^k e^{x_i - m_k} v_i$.

The update rule is perfectly symmetric to that of $d$, because both are essentially "a weighted sum of $e^{x_i - m}$":

$$
o_{k+1} = o_k \cdot e^{m_k - m_{k+1}} + e^{x_{k+1} - m_{k+1}} v_{k+1}
$$

After scanning all $N$ keys, the true attention output is $o_N / d_N$.


### 4.1 Hand-Calculating the Attention Output

Use a concrete query and three key-value pairs to verify the online flow. Set $d=1$ to simplify the numbers (in practice $d=64$ or $128$):

- $q = 1$, three keys $k = [3, 1, 2]$, corresponding values $v = [10, 20, 30]$
- attention scores $x_i = q \cdot k_i = [3, 1, 2]$ (omitting $\sqrt{d}$ since $d=1$)
- true output $o = \frac{e^3 \cdot 10 + e^1 \cdot 20 + e^2 \cdot 30}{e^3 + e^1 + e^2}$

Walk through the online flow:

| Step | $x_{k+1}$ | $v_{k+1}$ | $m$ | rescale | $d$ | $o$ |
|:---|:---|:---|:---|:---|:---|:---|
| init | - | - | $-\infty$ | - | 0 | 0 |
| $k=1$ | 3 | 10 | 3 | 0 | $e^0 = 1$ | $e^0 \cdot 10 = 10$ |
| $k=2$ | 1 | 20 | 3 | 1 | $1 + e^{-2} = 1.135$ | $10 + e^{-2} \cdot 20 = 10 + 2.71 = 12.71$ |
| $k=3$ | 2 | 30 | 3 | 1 | $1.135 + e^{-1} = 1.503$ | $12.71 + e^{-1} \cdot 30 = 12.71 + 11.04 = 23.75$ |

Final $o / d = 23.75 / 1.503 = 15.80$. Computing directly from the definition: $\frac{20.09 \cdot 10 + 2.72 \cdot 20 + 7.39 \cdot 30}{30.20} = \frac{200.9 + 54.4 + 221.7}{30.20} = \frac{477.0}{30.20} = 15.80$. The two match.


In [ ]:
def online_attention(q, K, V):
    """
    Single-pass attention, each query processed independently

    Args:
        q: a single query vector, shape [d]
        K: all keys, shape [N, d]
        V: all values, shape [N, d]
    Returns:
        attention output, shape [d]
    """
    d_head = q.shape[0]
    N = K.shape[0]

    m = float('-inf')       # running max
    d = 0.0                 # running denominator
    o = torch.zeros_like(q) # running output

    for i in range(N):
        # dot product of current key with query (divide by sqrt(d) for stable gradients)
        x_i = torch.dot(q, K[i]).item() / math.sqrt(d_head)
        v_i = V[i]

        m_new = max(m, x_i)
        rescale = math.exp(m - m_new)

        d = d * rescale + math.exp(x_i - m_new)
        o = o * rescale + math.exp(x_i - m_new) * v_i
        m = m_new

    return o / d

# Hand-computed example: d=1, q=1, k=[3,1,2], v=[10,20,30]
q = torch.tensor([1.0])
K = torch.tensor([[3.0], [1.0], [2.0]])
V = torch.tensor([[10.0], [20.0], [30.0]])
print("online attention output:", online_attention(q, K, V).item())
print("hand-computed result: 15.80")

# Compare against the standard implementation
def standard_attention(q, K, V):
    d_head = q.shape[0]
    scores = (K @ q) / math.sqrt(d_head)
    weights = torch.softmax(scores, dim=0)
    return weights @ V

print("standard attention output:", standard_attention(q, K, V).item())


## 5. Tiling: Putting the Single-Pass Algorithm on the GPU

At this point, we can compute attention with a single-pass loop on the CPU. But GPUs do not work this way — they rely on massive parallelism, and looping over tokens one by one is the worst possible form.

FlashAttention's second core trick is tiling: split Q, K, V into blocks, make each block just fit into SRAM, and move the loop to the block level.

### 5.1 Block Structure

Let the sequence length be $N$, the head dimension $d$, and the block size $B$ (commonly 64 or 128). Split the sequence into $N / B$ blocks:

- The outer loop iterates over Q blocks: $Q_1, Q_2, \dots, Q_{N/B}$, each of shape $[B, d]$
- The inner loop iterates over K, V blocks: $K_1, K_2, \dots, K_{N/B}$, each of shape $[B, d]$

For each Q block, independently maintain a set of running variables $(m, d, o)$, with shapes $[B]$, $[B]$, and $[B, d]$ respectively. Inside the inner loop, every time a pair of $(K_j, V_j)$ blocks is loaded, the variables are updated.

```text
for each Q block i (outer loop, parallelizable across blocks):
    m_i = -inf, d_i = 0, o_i = 0   # shape [B], [B], [B, d]
    for each K, V block j (inner loop):
        load K_j, V_j from HBM to SRAM
        S = Q_i @ K_j^T            # [B, B], exists only in SRAM
        m_block = S.max(dim=-1)    # [B]
        m_new = max(m_i, m_block)
        P = exp(S - m_new[:, None])
        d_i = d_i * exp(m_i - m_new) + P.sum(dim=-1)
        o_i = o_i * exp(m_i - m_new)[:, None] + P @ V_j
        m_i = m_new
    write o_i / d_i back to HBM   # write only the final result
```

### 5.2 Why Tiling Does Not Break Correctness

The update rule of online softmax is **associative**: accumulating within a block first and then rescaling with the block-level max gives the same result as scanning all elements at once. The reason is that the update rule depends only on the three quantities $(m, d, o)$, so block boundaries have no effect on the final result.

Intuitively: splitting $N$ keys into two halves, computing $(m_1, d_1, o_1)$ and $(m_2, d_2, o_2)$ separately, then rescaling and merging with the global max $m = \max(m_1, m_2)$ is equivalent to a single full scan. This is the mathematical guarantee behind block-level online softmax.


### 5.3 How to Choose the Block Size

The block size $B$ determines SRAM usage. Each block needs to hold $Q_i$ ($B \times d$), $K_j$ ($B \times d$), $V_j$ ($B \times d$), plus the intermediate $S$ and $P$ (each $B \times B$). The total SRAM footprint is roughly $3 B d + 2 B^2$ elements.

The A100's SRAM is about 19 MB (shared across multiple SMs). With $d = 128$, $B = 128$ is a common choice, occupying roughly $3 \cdot 128 \cdot 128 + 2 \cdot 128^2 = 81920$ FP16 elements, about 160 KB. Each SM processes multiple blocks concurrently, and the whole thing just fits in SRAM.

If $B$ is too small: too many loop iterations and high kernel launch overhead. If $B$ is too large: it no longer fits in SRAM and is forced to fall back to HBM. In production implementations, $B$ is typically determined by the GPU model and head dimension together, and the user does not need to tune it.


## 6. Why It Is Exact, Not Approximate

This section clears up a common misconception: FlashAttention does not sacrifice precision.

Sparse attention (such as Longformer, Big Bird) actively skips a subset of query-key pairs, on the grounds that "long-range attention is unimportant." This is a **structural approximation** — the output is mathematically different from dense attention.

Linear attention (Performer, Linear Transformer) uses the kernel trick to replace $QK^\top$ with a factorizable kernel function, reducing $O(N^2)$ to $O(N)$. This is also a **mathematical approximation**.

FlashAttention is different. It merely **rearranges the order of computation**:

- dense version: compute the full $S$ first, then softmax, then multiply by $V$
- flash version: compute a small block of $S$, immediately softmax it, immediately multiply by the corresponding block of $V$, and accumulate into a running sum

The mathematical operations of the two are completely equivalent — only the locations where intermediate results appear differ. FlashAttention never writes the full $S$ to HBM, yet every $S_{ij}$ participates in the computation. The difference in the final output is at the level of floating point error (around $10^{-6}$).

This is why FlashAttention can seamlessly replace standard attention — the training curve, model accuracy, and convergence behavior all stay the same.


## 7. A Simplified PyTorch Implementation

Next, we implement a teaching version of FlashAttention in pure PyTorch. **It does not aim for performance** (Python loops are far slower than CUDA); it aims only for mathematical correctness — so the reader can see exactly what every step of online softmax plus tiling is doing.

Implementation notes:

- Single head, batch=1 for simplicity (multiple heads are just an outer loop)
- Outer loop over Q blocks, inner loop over K/V blocks
- Use `torch.einsum` or `@` for the block-level matrix multiplies
- Explicitly maintain the three running variables $(m, d, o)$


In [ ]:
def flash_attention_forward(Q, K, V, block_size=4, causal=False):
    """
    Teaching version of FlashAttention forward, single head, batch=1

    Args:
        Q: shape [N, d]
        K: shape [N, d]
        V: shape [N, d]
        block_size: block size for Q/K/V
        causal: whether to apply a causal mask (lower triangular)
    Returns:
        O: shape [N, d]
    """
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    # The output and running variables live on HBM (simplification for the teaching version)
    # A real FlashAttention places them in SRAM
    O = torch.zeros(N, d)

    # Outer loop: each Q block is processed independently
    n_blocks = (N + block_size - 1) // block_size
    for i in range(n_blocks):
        q_start = i * block_size
        q_end = min(q_start + block_size, N)
        Qi = Q[q_start:q_end] * scale  # [B_q, d], multiply by scale up front

        # Running variables for this Q block
        B_q = q_end - q_start
        m_i = torch.full((B_q,), float('-inf'))
        d_i = torch.zeros(B_q)
        o_i = torch.zeros(B_q, d)

        # Inner loop: iterate over all K/V blocks
        for j in range(n_blocks):
            k_start = j * block_size
            k_end = min(k_start + block_size, N)
            Kj = K[k_start:k_end]  # [B_k, d]
            Vj = V[k_start:k_end]  # [B_k, d]

            # Compute this block's attention scores (exist only in SRAM)
            S = Qi @ Kj.transpose(-2, -1)  # [B_q, B_k]

            # Causal mask: set positions where query position < key position to -inf
            if causal:
                q_idx = torch.arange(q_start, q_end)[:, None]
                k_idx = torch.arange(k_start, k_end)[None, :]
                mask = k_idx > q_idx
                S = S.masked_fill(mask, float('-inf'))

            # online softmax update
            m_block = S.max(dim=-1).values      # [B_q]
            m_new = torch.maximum(m_i, m_block)

            # rescale the old accumulator
            alpha = torch.exp(m_i - m_new)      # [B_q]
            # exp of the new block (subtract new max for numerical stability)
            P = torch.exp(S - m_new[:, None])   # [B_q, B_k]

            d_i = d_i * alpha + P.sum(dim=-1)
            o_i = o_i * alpha[:, None] + P @ Vj
            m_i = m_new

        # Final output for this Q block
        O[q_start:q_end] = o_i / d_i[:, None]

    return O

# Small example: seq=8, d=4, friendly to hand computation
torch.manual_seed(42)
N, d = 8, 4
Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

O_flash = flash_attention_forward(Q, K, V, block_size=4)
print("FlashAttention output (first 3 rows):")
print(O_flash[:3])


## 8. Cross-Checking

Just implementing it is not enough — we need to prove it is numerically identical to standard attention. We compare against two reference implementations:

1. A hand-written standard attention (which explicitly constructs the $N \times N$ matrix)
2. PyTorch's official `scaled_dot_product_attention` (which automatically selects a FlashAttention backend under the hood)


In [ ]:
def standard_attention(Q, K, V, causal=False):
    """Standard attention, explicitly constructs the N×N matrix"""
    N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    S = (Q @ K.T) * scale  # [N, N]
    if causal:
        mask = torch.triu(torch.ones(N, N), diagonal=1).bool()
        S = S.masked_fill(mask, float('-inf'))
    P = torch.softmax(S, dim=-1)
    return P @ V

# Non-causal case
O_std = standard_attention(Q, K, V, causal=False)
O_sdpa = F.scaled_dot_product_attention(
    Q[None], K[None], V[None], is_causal=False
)[0]

print("=== Non-causal case ===")
print(f"Flash vs standard   max diff: {(O_flash - O_std).abs().max().item():.2e}")
print(f"Flash vs SDPA       max diff: {(O_flash - O_sdpa).abs().max().item():.2e}")
print(f"Standard vs SDPA    max diff: {(O_std - O_sdpa).abs().max().item():.2e}")

# Causal case
O_flash_causal = flash_attention_forward(Q, K, V, block_size=4, causal=True)
O_std_causal = standard_attention(Q, K, V, causal=True)
O_sdpa_causal = F.scaled_dot_product_attention(
    Q[None], K[None], V[None], is_causal=True
)[0]

print()
print("=== Causal case ===")
print(f"Flash vs standard   max diff: {(O_flash_causal - O_std_causal).abs().max().item():.2e}")
print(f"Flash vs SDPA       max diff: {(O_flash_causal - O_sdpa_causal).abs().max().item():.2e}")

print()
print("All differences are at the 1e-6 level — within floating point error, so FlashAttention is mathematically equivalent.")


In [ ]:
# Larger test: seq=256, d=64, to confirm accuracy on longer sequences
torch.manual_seed(0)
N, d = 256, 64
Q_big = torch.randn(N, d)
K_big = torch.randn(N, d)
V_big = torch.randn(N, d)

O_flash_big = flash_attention_forward(Q_big, K_big, V_big, block_size=64)
O_sdpa_big = F.scaled_dot_product_attention(
    Q_big[None], K_big[None], V_big[None]
)[0]

max_diff = (O_flash_big - O_sdpa_big).abs().max().item()
mean_diff = (O_flash_big - O_sdpa_big).abs().mean().item()
print(f"seq=256, d=64:")
print(f"  max diff:  {max_diff:.2e}")
print(f"  mean diff: {mean_diff:.2e}")
print(f"  passes 1e-5 threshold: {max_diff < 1e-5}")

# Also check the effect of block size on the result (should have no effect)
print()
print("Effect of block size on numerical correctness (all should be < 1e-5):")
for bs in [16, 32, 64, 128]:
    O_bs = flash_attention_forward(Q_big, K_big, V_big, block_size=bs)
    diff = (O_bs - O_sdpa_big).abs().max().item()
    print(f"  block_size={bs:>3d}: max diff {diff:.2e}")


## 9. Special Considerations for MoE

The attention component of a MoE (Mixture of Experts) model is exactly the same as in a dense model — the self-attention inside each expert is still standard causal multi-head attention, and FlashAttention is used the same way.

But MoE introduces a new problem: **expert routing produces many small-batch attention calls**.

### 9.1 The MoE Attention Data Flow

Let batch size be $B$, sequence length $S$, number of experts $E$, and top-k routing be $k$. Each token is routed to $k$ experts, so each expert receives on average $B \cdot S \cdot k / E$ tokens.

- $B=1, S=4096, k=2, E=8$: each expert receives about 1024 tokens on average
- $B=1, S=4096, k=1, E=64$: each expert receives about 64 tokens on average

These tokens come from different positions in the original sequence and **cannot be directly concatenated into a contiguous sequence for attention** — their causal relationship is relative to the original sequence, not to their order inside the expert.

### 9.2 Two Ways to Handle It

The first is **padding**: each expert maintains a $B \times S$ tensor internally; tokens routed to that expert stay put and the rest are zero-filled. Attention is still computed on $B \times S$, but only $B \cdot S \cdot k / E$ tokens are valid. This causes massive waste — compute and memory are billed by $B \times S$, but only a $k/E$ fraction actually contributes.

The second is **grouped GEMM / grouped attention**: gather each expert's non-contiguous tokens into a variable-length sequence and let the attention kernel handle variable-length input directly. The variable-length variant of FlashAttention (varlen) supports this mode and avoids padding.

| Approach | Compute / memory waste | Implementation complexity |
|:---|:---|:---|
| Padding | $E/k$ times | Simple |
| Grouped (varlen) | Near zero | Needs a custom kernel |

### 9.3 Impact on FlashAttention

The FlashAttention algorithm itself needs no special handling under MoE — as long as it is given a correct $(Q, K, V)$ triple, it computes along the standard flow. The difference is **how the input is organized**:

- dense model: one large tensor of shape $[B, N\_\text{heads}, S, d]$, one call
- MoE padding: one tensor of shape $[B, N\_\text{heads}, S, d]$ per expert, $E$ calls, most of each is padding
- MoE varlen: concatenate all experts' valid tokens into one long 1D tensor, plus each expert's cu_seqlens (cumulative lengths), one call

varlen is the recommended approach for MoE plus FlashAttention. Inference frameworks such as Megatron-LM, vLLM, and SGLang all implement grouped attention on top of varlen.


In [ ]:
# Quantify the waste of padding under MoE
def moe_padding_waste(B, S, E, k, n_heads, d):
    """Compute the effective compute ratio of MoE under padding mode"""
    # Average number of tokens each expert receives
    avg_tokens_per_expert = B * S * k / E
    # Under padding mode, each expert still computes on [B, S]
    actual_tokens = B * S
    # Effective ratio
    eff_ratio = avg_tokens_per_expert / actual_tokens
    # Total FLOPs (only counting the attention score matmul)
    flops_padding = E * B * n_heads * S * S * d * 2
    flops_useful = E * avg_tokens_per_expert * n_heads * S * d * 2
    return eff_ratio, flops_padding, flops_useful

print(f"{'config':>30}  {'valid token ratio':>18}  {'waste factor':>12}")
print("-" * 65)
configs = [
    ("dense (E=1, k=1)", 1, 4096, 1, 1, 32, 128),
    ("MoE E=8, k=2", 1, 4096, 8, 2, 32, 128),
    ("MoE E=64, k=1", 1, 4096, 64, 1, 32, 128),
    ("MoE E=64, k=2", 1, 4096, 64, 2, 32, 128),
    ("MoE E=128, k=2", 1, 4096, 128, 2, 32, 128),
]
for name, B, S, E, k, nh, d in configs:
    eff, _, _ = moe_padding_waste(B, S, E, k, nh, d)
    waste = 1 / eff if eff > 0 else float('inf')
    print(f"{name:>30}  {eff:>17.1%}  {waste:>11.1f}x")

print()
print("At E=128, k=2, the effective compute under padding mode is only 1.6% — 64x waste.")
print("The varlen mode sidesteps this problem entirely.")


## Appendix: The Evolution of FlashAttention

### A7.1 FlashAttention-2: Fewer Non-matmul FLOPs

FlashAttention-1's forward pass was already fast, but the backward pass and work partitioning still had room for improvement. FlashAttention-2 (Dao 2023) targets three optimizations.

First, **fewer non-matmul FLOPs**. On a GPU, matmul (tensor core) throughput is far higher than that of other operations (such as rescaling and softmax). FA-1 had some non-matmul operations in the inner loop; FA-2 rearranges them so that tensor core utilization is higher.

Second, **better parallelization**. FA-1 parallelizes mainly across the batch and head dimensions, leaving the GPU underutilized when sequences are long but the batch is small. FA-2 also splits the sequence dimension — different Q blocks are assigned to different SMs, significantly improving throughput for long-sequence scenarios.

Third, **work partitioning**. FA-1's inner/outer loop partitioning forced frequent synchronization between warps on the GPU. FA-2 redesigns this so each warp processes a block independently, reducing synchronization overhead.

In practice, FA-2 is about 2x faster than FA-1, reaching 50-70% of the theoretical matmul throughput on the A100.

### A7.2 FlashAttention-3: Hopper Architecture and FP8

FlashAttention-3 (Shah et al. 2024) is designed specifically for the Hopper architecture (H100), taking advantage of three new features.

First, **WGMMA (warp-group matrix multiply accumulate)**. An asynchronous tensor core instruction introduced in Hopper, allowing matmul to overlap with other operations. FA-3 schedules softmax and rescaling into the gaps of the matmul, almost completely hiding the non-matmul overhead.

Second, **TMA (tensor memory accelerator)**. Hopper's hardware DMA can move data asynchronously between HBM and SRAM without occupying the SM. FA-3 uses TMA for double buffering — the next K/V block is prefetched while the current block is being computed.

Third, **FP8 tensor cores**. Hopper supports FP8 (E4M3 / E5M2) tensor core matmul, with twice the throughput of FP16. FA-3 computes $QK^\top$ and $PV$ in FP8 in the forward pass, but still accumulates the softmax in FP32 to preserve numerical stability.

In practice, FA-3 reaches 1.5-2 PFLOPs (FP8) on the H100, approaching 75% of the hardware's theoretical upper bound.

### A7.3 A Guide to the Triton Implementation

OpenAI's Triton is a high-level language for writing GPU kernels in Python, more readable than CUDA. The FlashAttention project maintains an official Triton implementation, which is the best reference for learning about tiled kernels.

The source lives in the `flash_attn/flash_attn_triton.py` file of the [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) repository. The core structure is a function decorated with `@triton.jit`; the outer Q block loop uses `tl.range`, the inner K/V block loop is also `tl.range`, and each block is read with `tl.load` from HBM and written back with `tl.store`. The online softmax updates of $(m, d, o)$ translate directly into a few lines of Triton code, exactly mirroring the PyTorch version implemented in this appendix.

Reading advice: first build intuition from the PyTorch implementation in this appendix, then cross-reference the Triton version to understand the GPU programming details (shared memory, warp synchronization, block pointer advancement). The official Triton tutorial also has a simplified [Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html) version that is more readable than the Dao official version.


### References

- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), NeurIPS 2022
- Dao, [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691), 2023
- Shah et al., [FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-precision](https://arxiv.org/abs/2407.08608), 2024
- Milakov & Gimelshein, [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867), 2018 (the original online softmax paper)
- Tillet et al., [Triton: an intermediate language and compiler for tiled neural network computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox), 2019


## Summary

- [ ] A GPU has two tiers of memory: HBM is large but slow (~2 TB/s), SRAM is small but fast (~19 TB/s); attention is the prototypical memory-bound operator
- [ ] Standard attention writes each $N \times N$ intermediate matrix to HBM 4-5 times; once sequences grow long, the transfer volume far exceeds the compute volume
- [ ] Naive softmax overflows (under FP32, $e^{88}$ is close to inf); stable softmax that subtracts the max is the baseline defense
- [ ] The logsumexp trick turns $\log \sum e^{x_i}$ into $x_\max + \log \sum e^{x_i - x_\max}$, avoiding both $\log(0)$ and $\exp$ overflow
- [ ] Online softmax fuses "find max" and "sum" into a single pass, using running $(m, d)$ and the $e^{m_\text{old} - m_\text{new}}$ rescale to produce an equivalent computation
- [ ] The attention output $o = \sum p_i v_i$ can likewise be accumulated in a single pass via the running $o_k$, with a rule symmetric to that of $d$
- [ ] Tiling splits Q, K, V into blocks that fit in SRAM; block-level online softmax stays correct because the update rule is associative
- [ ] FlashAttention is exact, not approximate — it merely reorders the computation, which is fundamentally different from sparse / linear attention
- [ ] Under MoE the attention itself is unchanged, but expert routing produces many small batches; padding mode wastes heavily, while the varlen mode is far better
- [ ] FlashAttention-2 optimizes work partitioning; FlashAttention-3 takes advantage of Hopper's WGMMA / TMA / FP8


## Homework

The following three exercises help you consolidate online softmax and the core update rules of attention. You may ask an AI for ideas, break the problem into steps, or sanity-check your direction, but it is not advisable to have an AI "just do this problem for you."

**Homework 1: Implement the running max update**

Below is a simplified online softmax skeleton, missing the running max and rescale parts. Fill in the three lines `m_new`, `rescale`, and `d` so that the output is exactly equal to `torch.softmax`.

```python
def online_softmax_homework(x):
    m = float('-inf')
    d = 0.0
    for xi in x:
        xi = xi.item()
        # TODO: fill in the three lines
        # 1. m_new = ?
        # 2. rescale = ?
        # 3. d = ?
        m = m_new
    return torch.tensor([math.exp(xi - m) / d for xi in x])

x = torch.tensor([1.0, 5.0, 3.0, 7.0, 2.0])
result = online_softmax_homework(x)
expected = torch.softmax(x, dim=0)
assert torch.allclose(result, expected, atol=1e-6), f"diff too large: {(result - expected).abs().max()}"
print("Homework 1 passed: online softmax implementation is correct")
```

Hint: refer to the update rule in section 3.1 — the new max is the larger of the old max and the current value; the rescale factor is $\exp(m_\text{old} - m_\text{new})$; the new denominator is "old denominator times rescale" plus "the new term $\exp(x_i - m_\text{new})$".

**Homework 2: Verify the numerical equivalence of online attention**

Complete the function below so that, for random inputs $(Q, K, V)$, its output differs from `F.scaled_dot_product_attention` by less than $10^{-5}$.

```python
def online_attention_homework(q, K, V):
    """Single-query attention"""
    d_head = q.shape[0]
    m = float('-inf')
    d = 0.0
    o = torch.zeros_like(q)

    for i in range(K.shape[0]):
        x_i = (q * K[i]).sum().item() / math.sqrt(d_head)
        v_i = V[i]
        # TODO: fill in the four update steps m_new, rescale, d, o
        # 1. m_new = ?
        # 2. rescale = ?
        # 3. d = ?
        # 4. o = ?
        m = m_new

    return o / d

torch.manual_seed(0)
d = 8
N = 16
q = torch.randn(d)
K = torch.randn(N, d)
V = torch.randn(N, d)
out = online_attention_homework(q, K, V)
ref = F.scaled_dot_product_attention(q[None, None], K[None], V[None])[0, 0]
assert torch.allclose(out, ref, atol=1e-5), f"diff: {(out - ref).abs().max()}"
print("Homework 2 passed: online attention is numerically equivalent")
```

Hint: the update rule for $o$ is perfectly symmetric to that of $d$ — both are "old value times the rescale factor" plus "$\exp(x_i - m_\text{new})$ times the corresponding $v_i$". Refer to the formulas in section 4.

**Homework 3: Estimate MoE Padding Waste**

A MoE model has $E=64$ experts, top-1 routing ($k=1$), sequence length $S=8192$, and batch $B=4$. Under padding mode, each expert's attention call still processes a $[B, S]$ tensor.

(1) On average, how many valid tokens does each expert receive? (2) Under padding mode, what is the ratio of total valid tokens to total tokens processed across all experts? (3) After replacing padding with varlen, how many times the attention compute is saved?

```python
B, S, E, k = 4, 8192, 64, 1
total_tokens = B * S
# TODO: fill in the three lines
# tokens_per_expert = ?
# efficiency = ?       # valid / total processed
# speedup_varlen = ?   # speedup of varlen over padding

assert tokens_per_expert == 512, f"tokens_per_expert should be 512, you got {tokens_per_expert}"
assert abs(efficiency - 1/64) < 1e-6, f"efficiency should be 1/64"
assert abs(speedup_varlen - 64) < 1e-6, f"speedup_varlen should be 64"
print("Homework 3 passed: understood MoE padding waste")
```

Hint: each token is routed to $k$ experts, so each expert receives $B \cdot S \cdot k / E$ tokens on average. Under padding mode, the expert still processes the full $B \cdot S$ tensor, so the efficiency is the ratio of the two. Varlen concatenates the valid tokens and computes on them, so the speedup is the reciprocal of the efficiency.
